In [ ]:
# | default_exp production.cleaning

# Pre-clustering classification cleaning

> Canonicalisation of raw Zooniverse / Panoptes classification data — drops dirt rows, normalises angle ranges, fixes ellipse axis ordering, and derives angular components for clustering. **Not** the catalog reduction (= clustering) pipeline; this runs upstream of L0→L1A.

In [ ]:
# | export
"""production.cleaning — pre-clustering canonicalisation of raw classifications.

Run upstream of the catalog reduction (clustering) pipeline. Inputs are
raw per-classification marking rows (one row per fan/blotch a volunteer
drew); outputs are the same rows after dropping dirt, fixing geometry
conventions, and adding the angular features DBSCAN consumes.

Two raw sources are supported via the ``source=`` argument of the top-level
:func:`clean_classifications` orchestrator:

* ``"zooniverse_v1"`` — the legacy interface used through v3.x. Has
  characteristic default-marking idiosyncrasies (auto-spawned 10x10
  ellipses at origin, default-spread fans, etc.) that need filtering.
* ``"panoptes"`` — the modern Panoptes-based interface (workflow 12978).
  Empirical 200k-row check on the 2026-04-23 export shows the default-
  spawn idiosyncrasies do **not** survive in this UI; only NaN sweeps and
  out-of-frame filtering still pay rent.

The individual filter / canonicalisation steps are also exported for
direct use (and for reproducibility of v3.x via the legacy path).
"""
from typing import Literal

import numpy as np
import pandas as pd

# Tile geometry — shared with production.coverage
TILE_WIDTH_PX = 840
TILE_HEIGHT_PX = 648

DEFAULT_OUT_OF_FRAME_TOLERANCE_PX = 25
"""How far outside the tile a marking centre may sit before it's discarded."""

FAN_DATA_COLS = ["x", "y", "distance", "angle", "spread"]
BLOTCH_DATA_COLS = ["x", "y", "radius_1", "radius_2"]

Source = Literal["zooniverse_v1", "panoptes"]


## NaN sweep — drop incomplete markings

In [ ]:
# | export
def filter_nan_required(df: pd.DataFrame) -> pd.DataFrame:
    """Drop fans/blotches missing any required column.

    Required cols: fans need ``x, y, distance, angle, spread``; blotches
    need ``x, y, radius_1, radius_2``. Rows with ``marking != fan|blotch``
    pass through untouched.
    """
    fans = df[df.marking == "fan"].dropna(how="any", subset=FAN_DATA_COLS)
    blotches = df[df.marking == "blotch"].dropna(how="any", subset=BLOTCH_DATA_COLS)
    rest = df[~df.marking.isin(["fan", "blotch"])]
    return pd.concat([fans, blotches, rest], ignore_index=True)


## Default-marking filter (Zooniverse v1 only)

In [ ]:
# | export
def filter_default_markings(df: pd.DataFrame) -> pd.DataFrame:
    """Drop the legacy Zooniverse-v1 auto-spawned default markings.

    Verbatim port of legacy ``planet4.reduction.filter_data`` steps 2-5:

    * Origin-pinned default fan: ``|x|<eps & |y|<eps & |angle|<eps & distance~10``.
    * Second-default fan: ``|angle|~90 & spread~2.017450 & distance~10``.
    * Origin-pinned 10x10 default ellipse blotch: ``|x|<eps & |y|<eps & r1~10 & r2~10``.
    * Origin-pinned ``none`` row.

    Empirically a no-op for Panoptes-12978 data (the new UI does not auto-spawn
    these defaults on click-without-drag); kept available for legacy reprocessing.
    """
    eps = 1e-5
    fans = df[df.marking == "fan"]
    blotches = df[df.marking == "blotch"]
    rest = df[~df.marking.isin(["fan", "blotch"])]

    # default fan (origin)
    fzero = (fans.x.abs() < eps) & (fans.y.abs() < eps)
    fdef = (fans.angle.abs() < eps) & ((fans.distance - 10).abs() < eps)
    fans = fans[~(fzero & fdef)]

    # default fan (second variant)
    fdef2 = (
        ((fans.angle.abs() - 90.0) < eps)
        & ((fans.spread - 2.017450) < eps)
        & ((fans.distance - 10) < eps)
    )
    fans = fans[~fdef2]

    # default blotch (10x10 ellipse at origin)
    bzero = (blotches.x.abs() < eps) & (blotches.y.abs() < eps)
    bdef = ((blotches.radius_1 - 10) < eps) & ((blotches.radius_2 - 10).abs() < eps)
    blotches = blotches[~(bzero & bdef)]

    # default 'none' rows at origin
    rzero = (rest.x.abs() < eps) & (rest.y.abs() < eps)
    rest = rest[~rzero]

    return pd.concat([fans, blotches, rest], ignore_index=True)


## Out-of-frame filter — drop markings far outside tile

In [ ]:
# | export
def filter_out_of_frame(
    df: pd.DataFrame,
    *,
    tolerance_px: int = DEFAULT_OUT_OF_FRAME_TOLERANCE_PX,
) -> pd.DataFrame:
    """Drop markings whose centre falls more than ``tolerance_px`` outside the
    840x648 tile. ``marking == "none"`` rows are exempted (they record
    'volunteer saw nothing' and have meaningless coords).
    """
    none_rows = df[df.marking == "none"]
    rest = df[df.marking != "none"]
    keep = (
        (rest.x >= -tolerance_px)
        & (rest.x <= TILE_WIDTH_PX + tolerance_px)
        & (rest.y >= -tolerance_px)
        & (rest.y <= TILE_HEIGHT_PX + tolerance_px)
    )
    return pd.concat([rest[keep], none_rows], ignore_index=True)


## Blotch geometry canonicalisation

In [ ]:
# | export
def canonicalize_blotch_geometry(df: pd.DataFrame) -> pd.DataFrame:
    """Make blotch ellipses canonical: ``radius_1 >= radius_2``, angle in ``[0, 180)``.

    Verbatim port of legacy ``planet4.reduction.convert_ellipse_angles``:
    where ``radius_1 < radius_2`` we swap the radii **and** add 90 deg to the
    angle, then take ``angle % 180`` (ellipse symmetry).

    Modifies the dataframe in place and returns it.
    """
    blotch_idx = df.marking == "blotch"
    swap_idx = blotch_idx & (df.radius_1 < df.radius_2)

    # Swap r1 <-> r2 where needed
    df.loc[swap_idx, ["radius_1", "radius_2"]] = df.loc[
        swap_idx, ["radius_2", "radius_1"]
    ].values

    # Add 90 deg only on the swap rows
    df.loc[swap_idx, "angle"] = df.loc[swap_idx, "angle"] + 90.0

    # Fold all blotch angles into [0, 180)
    df.loc[blotch_idx, "angle"] = df.loc[blotch_idx, "angle"] % 180.0
    return df


## Fan angle canonicalisation

In [ ]:
# | export
def canonicalize_fan_angles(df: pd.DataFrame) -> pd.DataFrame:
    """Fold fan angles into ``[0, 360)``.

    Verbatim port of legacy ``planet4.reduction.normalize_fan_angles``.
    Empirically a no-op for Panoptes-12978 (the UI already produces fan
    angles in ``[0, 360)``); kept for legacy reprocessing.
    """
    fan_idx = df.marking == "fan"
    df.loc[fan_idx, "angle"] = df.loc[fan_idx, "angle"] % 360.0
    return df


## Angular components for clustering

In [ ]:
# | export
def compute_angle_components(df: pd.DataFrame) -> pd.DataFrame:
    """Add ``x_angle = cos(deg2rad(angle))`` and ``y_angle = sin(deg2rad(angle))``.

    These are the angular features the catalog reduction (clustering) reads
    directly: ``production.dbscan`` clusters fans on ``(x_angle, y_angle)``
    and blotches on ``y_angle``. Must run **after** the blotch and fan angle
    canonicalisations so the components reflect the canonical-quadrant angle.
    """
    rad = np.deg2rad(df["angle"].astype("float64"))
    df["x_angle"] = np.cos(rad)
    df["y_angle"] = np.sin(rad)
    return df


## Orchestrator — dispatch per raw source

In [ ]:
# | export
def clean_classifications(
    df: pd.DataFrame,
    *,
    source: Source = "panoptes",
    out_of_frame_tolerance_px: int = DEFAULT_OUT_OF_FRAME_TOLERANCE_PX,
) -> pd.DataFrame:
    """Top-level orchestrator. Dispatches the right cleanup steps per raw source.

    Steps run, in order:

    1. ``filter_nan_required`` (always)
    2. ``filter_default_markings``  (zooniverse_v1 only)
    3. ``filter_out_of_frame`` (always)
    4. ``canonicalize_blotch_geometry`` (always)
    5. ``canonicalize_fan_angles`` (zooniverse_v1 only — Panoptes already canonical)
    6. ``compute_angle_components`` (always)
    """
    if source not in ("zooniverse_v1", "panoptes"):
        raise ValueError(f"unknown source: {source!r}")

    df = filter_nan_required(df)
    if source == "zooniverse_v1":
        df = filter_default_markings(df)
    df = filter_out_of_frame(df, tolerance_px=out_of_frame_tolerance_px)
    df = canonicalize_blotch_geometry(df)
    if source == "zooniverse_v1":
        df = canonicalize_fan_angles(df)
    df = compute_angle_components(df)
    return df
